# 🍬 Nassau Candy Distributor
## Factory Reallocation & Shipping Optimization Recommendation System

---

**Project Domain:** Supply Chain Analytics  
**Dataset:** Nassau_Candy_Distributor.csv (10,194 orders)  
**Objective:** Predict shipping lead times, identify bottlenecks, and recommend optimal factory-product assignments to minimize lead time and maximize profitability.

---

### 📌 Project Structure
1. Library Imports & Setup
2. Data Loading & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Predictive Modeling (Lead Time Prediction)
6. Route & Product Clustering
7. Scenario Simulation Engine
8. Optimization & Recommendations
9. KPI Summary & Insights

---
## 📦 Section 1: Library Imports & Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans

print('✅ All libraries imported successfully!')

---
## 📂 Section 2: Data Loading & Preprocessing

In [ ]:
# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv('Nassau_Candy_Distributor.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

In [ ]:
# ── Data types & missing values ───────────────────────────
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
# ── Date parsing & Lead Time calculation ─────────────────
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)
df['Lead Time']  = (df['Ship Date'] - df['Order Date']).dt.days
df['Margin %']   = (df['Gross Profit'] / df['Sales'] * 100).round(2)
df['Month']      = df['Order Date'].dt.to_period('M').astype(str)

# ── Map each product to its current factory ───────────────
FACTORY_MAP = {
    'Wonka Bar - Nutty Crunch Surprise':  "Lot's O' Nuts",
    'Wonka Bar - Fudge Mallows':          "Lot's O' Nuts",
    'Wonka Bar -Scrumdiddlyumptious':     "Lot's O' Nuts",
    'Wonka Bar - Milk Chocolate':         "Wicked Choccy's",
    'Wonka Bar - Triple Dazzle Caramel':  "Wicked Choccy's",
    'Laffy Taffy':           'Sugar Shack',
    'SweeTARTS':             'Sugar Shack',
    'Nerds':                 'Sugar Shack',
    'Fun Dip':               'Sugar Shack',
    'Fizzy Lifting Drinks':  'Sugar Shack',
    'Everlasting Gobstopper':'Secret Factory',
    'Hair Toffee':           'The Other Factory',
    'Lickable Wallpaper':    'Secret Factory',
    'Wonka Gum':             'Secret Factory',
    'Kazookles':             'The Other Factory'
}
df['Factory'] = df['Product Name'].map(FACTORY_MAP)

# Factory coordinates (lat/lon)
FACTORY_COORDS = {
    "Lot's O' Nuts":    (32.881893, -111.768036),
    "Wicked Choccy's":  (32.076176,  -81.088371),
    'Sugar Shack':      (48.119140,  -96.181150),
    'Secret Factory':   (41.446333,  -90.565487),
    'The Other Factory':(35.117500,  -89.971107)
}
ALL_FACTORIES = list(FACTORY_COORDS.keys())

print('✅ Preprocessing complete!')
print(f'   Lead Time range: {df["Lead Time"].min()} – {df["Lead Time"].max()} days')
df[['Order Date','Ship Date','Lead Time','Factory','Margin %']].head()

In [ ]:
# ── Descriptive statistics ────────────────────────────────
print('📊 Numerical Summary:')
df[['Sales','Gross Profit','Cost','Units','Lead Time','Margin %']].describe().round(2)

---
## 📊 Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── KPI Summary ───────────────────────────────────────────
print('=' * 50)
print('  KEY PERFORMANCE INDICATORS')
print('=' * 50)
print(f'  Total Orders     : {len(df):,}')
print(f'  Total Revenue    : ${df["Sales"].sum():,.2f}')
print(f'  Total Profit     : ${df["Gross Profit"].sum():,.2f}')
print(f'  Avg Lead Time    : {df["Lead Time"].mean():.0f} days')
print(f'  Avg Profit Margin: {df["Margin %"].mean():.1f}%')
print(f'  Unique Products  : {df["Product Name"].nunique()}')
print(f'  Unique Factories : {df["Factory"].nunique()}')
print(f'  Regions Served   : {df["Region"].nunique()}')

In [ ]:
# ── 3.1 Performance by Region ─────────────────────────────
region_summary = df.groupby('Region').agg(
    Orders       = ('Row ID', 'count'),
    Total_Sales  = ('Sales', 'sum'),
    Total_Profit = ('Gross Profit', 'sum'),
    Avg_LeadTime = ('Lead Time', 'mean'),
    Avg_Margin   = ('Margin %', 'mean')
).round(2).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4e79a7','#f28e2b','#e15759','#76b7b2']
axes[0].bar(region_summary['Region'], region_summary['Orders'], color=colors)
axes[0].set_title('Orders by Region', fontweight='bold')
axes[0].set_ylabel('Number of Orders')
for i, v in enumerate(region_summary['Orders']):
    axes[0].text(i, v+10, str(v), ha='center', fontweight='bold')

axes[1].bar(region_summary['Region'], region_summary['Avg_LeadTime'], color=colors)
axes[1].set_title('Avg Lead Time by Region', fontweight='bold')
axes[1].set_ylabel('Days')
axes[1].set_ylim(1280, 1340)
for i, v in enumerate(region_summary['Avg_LeadTime']):
    axes[1].text(i, v+0.5, f'{v:.0f}d', ha='center', fontweight='bold')

plt.suptitle('Regional Performance Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(region_summary.to_string(index=False))

In [ ]:
# ── 3.2 Factory Performance ───────────────────────────────
fac_summary = df.groupby('Factory').agg(
    Orders       = ('Row ID', 'count'),
    Total_Sales  = ('Sales', 'sum'),
    Total_Profit = ('Gross Profit', 'sum'),
    Avg_LeadTime = ('Lead Time', 'mean')
).round(2).reset_index().sort_values('Orders', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(fac_summary['Factory'], fac_summary['Orders'], color='#4e79a7')
axes[0].set_title('Order Volume by Factory', fontweight='bold')
axes[0].set_xticklabels(fac_summary['Factory'], rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Orders')

axes[1].bar(fac_summary['Factory'], fac_summary['Avg_LeadTime'], color='#e15759')
axes[1].set_title('Avg Lead Time by Factory', fontweight='bold')
axes[1].set_xticklabels(fac_summary['Factory'], rotation=25, ha='right', fontsize=9)
axes[1].set_ylabel('Days')
axes[1].set_ylim(1250, 1360)

axes[2].bar(fac_summary['Factory'], fac_summary['Total_Profit'], color='#59a14f')
axes[2].set_title('Total Profit by Factory', fontweight='bold')
axes[2].set_xticklabels(fac_summary['Factory'], rotation=25, ha='right', fontsize=9)
axes[2].set_ylabel('Profit ($)')

plt.suptitle('Factory Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(fac_summary.to_string(index=False))

In [ ]:
# ── 3.3 Ship Mode Analysis ────────────────────────────────
sm_summary = df.groupby('Ship Mode').agg(
    Orders       = ('Row ID', 'count'),
    Avg_LeadTime = ('Lead Time', 'mean'),
    Avg_Cost     = ('Cost', 'mean')
).round(2).reset_index().sort_values('Avg_LeadTime')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#59a14f' if v == sm_summary['Avg_LeadTime'].min() 
          else '#e15759' if v == sm_summary['Avg_LeadTime'].max() 
          else '#4e79a7' for v in sm_summary['Avg_LeadTime']]

axes[0].barh(sm_summary['Ship Mode'], sm_summary['Avg_LeadTime'], color=colors)
axes[0].set_title('Avg Lead Time by Ship Mode\n(🟢 Fastest | 🔴 Slowest)', fontweight='bold')
axes[0].set_xlabel('Days')
for i, v in enumerate(sm_summary['Avg_LeadTime']):
    axes[0].text(v+0.5, i, f'{v:.0f}d', va='center', fontweight='bold')

axes[1].bar(sm_summary['Ship Mode'], sm_summary['Orders'], color='#f28e2b')
axes[1].set_title('Order Volume by Ship Mode', fontweight='bold')
axes[1].set_ylabel('Orders')
for i, v in enumerate(sm_summary['Orders']):
    axes[1].text(i, v+10, str(v), ha='center', fontweight='bold')

plt.suptitle('Ship Mode Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('⚠️  Key Insight: Standard Class is FASTER than First Class — routing inefficiency!')
print(sm_summary.to_string(index=False))

In [ ]:
# ── 3.4 Product Analysis ──────────────────────────────────
prod_summary = df.groupby(['Product Name','Factory','Division']).agg(
    Orders       = ('Row ID', 'count'),
    Avg_Sales    = ('Sales', 'mean'),
    Avg_Profit   = ('Gross Profit', 'mean'),
    Avg_LeadTime = ('Lead Time', 'mean'),
    Avg_Margin   = ('Margin %', 'mean')
).round(2).reset_index().sort_values('Avg_LeadTime', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

lt_colors = ['#e15759' if v >= prod_summary['Avg_LeadTime'].quantile(0.75) 
             else '#59a14f' if v <= prod_summary['Avg_LeadTime'].quantile(0.25) 
             else '#4e79a7' for v in prod_summary['Avg_LeadTime']]
axes[0].barh(prod_summary['Product Name'], prod_summary['Avg_LeadTime'], color=lt_colors)
axes[0].set_title('Avg Lead Time by Product\n(🔴 High | 🟢 Low)', fontweight='bold')
axes[0].set_xlabel('Days')
axes[0].tick_params(labelsize=8)

prod_profit = prod_summary.sort_values('Avg_Profit', ascending=False)
p_colors = ['#f28e2b' if v >= prod_profit['Avg_Profit'].mean() else '#76b7b2' for v in prod_profit['Avg_Profit']]
axes[1].barh(prod_profit['Product Name'], prod_profit['Avg_Profit'], color=p_colors)
axes[1].set_title('Avg Profit per Order by Product', fontweight='bold')
axes[1].set_xlabel('Avg Profit ($)')
axes[1].axvline(prod_profit['Avg_Profit'].mean(), color='red', linestyle='--', label=f'Avg: ${prod_profit["Avg_Profit"].mean():.2f}')
axes[1].legend()
axes[1].tick_params(labelsize=8)

plt.suptitle('Product-Level Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Lead Time Distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Lead Time'], bins=40, color='#4e79a7', edgecolor='white')
axes[0].axvline(df['Lead Time'].mean(), color='red', linestyle='--', lw=2,
                label=f'Mean: {df["Lead Time"].mean():.0f}d')
axes[0].axvline(df['Lead Time'].median(), color='orange', linestyle='--', lw=2,
                label=f'Median: {df["Lead Time"].median():.0f}d')
axes[0].set_title('Lead Time Distribution', fontweight='bold')
axes[0].set_xlabel('Days'); axes[0].set_ylabel('Frequency')
axes[0].legend()

monthly = df.groupby('Month').agg(Orders=('Row ID','count'), Profit=('Gross Profit','sum')).reset_index()
ax2 = axes[1].twinx()
axes[1].bar(monthly['Month'], monthly['Orders'], color='#4e79a7', alpha=0.6, label='Orders')
ax2.plot(monthly['Month'], monthly['Profit'], 'o-', color='#e15759', lw=2, label='Profit')
n = len(monthly); step = max(1, n//8)
axes[1].set_xticks(range(0, n, step))
axes[1].set_xticklabels([monthly['Month'].iloc[i] for i in range(0, n, step)], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Monthly Orders & Profit Trend', fontweight='bold')
axes[1].set_ylabel('Orders'); ax2.set_ylabel('Profit ($)')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, labels1+labels2, loc='upper left')

plt.suptitle('Lead Time Distribution & Monthly Trends', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6 Correlation Heatmap ───────────────────────────────
corr_cols = ['Sales','Units','Gross Profit','Cost','Lead Time','Margin %']
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚙️ Section 4: Feature Engineering

In [ ]:
# ── Encode categorical variables ──────────────────────────
le_dict = {}
dfe = df.copy()

for col in ['Ship Mode','Region','Product Name','Factory','Division','Country/Region']:
    le = LabelEncoder()
    dfe[col+'_enc'] = le.fit_transform(dfe[col])
    le_dict[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

FEATURE_COLS = ['Ship Mode_enc','Region_enc','Product Name_enc',
                'Factory_enc','Division_enc','Units','Country/Region_enc']

print(f'\n✅ Feature matrix ready: {len(FEATURE_COLS)} features')
dfe[FEATURE_COLS + ['Lead Time']].head()

---
## 🤖 Section 5: Predictive Modeling — Lead Time Prediction

In [ ]:
# ── Train-test split ──────────────────────────────────────
X = dfe[FEATURE_COLS]
y = dfe['Lead Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples: {len(X_train):,}')
print(f'Testing  samples: {len(X_test):,}')

In [ ]:
# ── Train & evaluate 3 models ─────────────────────────────
models = {
    'Linear Regression':  LinearRegression(),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
trained_models = {}

print(f'{"Model":<22} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('-' * 50)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Preds': preds}
    trained_models[name] = model
    print(f'{name:<22} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f}')

best_name  = min(results, key=lambda x: results[x]['RMSE'])
best_model = trained_models[best_name]
print(f'\n✅ Best Model: {best_name}')

In [ ]:
# ── Model comparison chart ────────────────────────────────
model_names = list(results.keys())
rmse_vals   = [results[m]['RMSE'] for m in model_names]
mae_vals    = [results[m]['MAE']  for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#59a14f' if n == best_name else '#4e79a7' for n in model_names]
axes[0].bar(model_names, rmse_vals, color=colors)
axes[0].set_title('RMSE Comparison (lower = better)\n🟢 Best Model', fontweight='bold')
axes[0].set_ylabel('RMSE')
for i, v in enumerate(rmse_vals):
    axes[0].text(i, v+0.5, f'{v:.2f}', ha='center', fontweight='bold')

axes[1].bar(model_names, mae_vals, color=colors)
axes[1].set_title('MAE Comparison (lower = better)\n🟢 Best Model', fontweight='bold')
axes[1].set_ylabel('MAE')
for i, v in enumerate(mae_vals):
    axes[1].text(i, v+0.5, f'{v:.2f}', ha='center', fontweight='bold')

plt.suptitle('Predictive Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔵 Section 6: Route & Product Clustering

In [ ]:
# ── KMeans clustering on routes ───────────────────────────
route_features = ['Lead Time','Sales','Gross Profit','Cost','Units']
route_data     = df[route_features].copy()

# Normalize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(route_data)

# Fit KMeans with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Cluster profiles
cluster_profile = df.groupby('Cluster').agg(
    Count        = ('Row ID', 'count'),
    Avg_LeadTime = ('Lead Time', 'mean'),
    Avg_Sales    = ('Sales', 'mean'),
    Avg_Profit   = ('Gross Profit', 'mean')
).round(2)

print('Cluster Profiles:')
print(cluster_profile)

# Label clusters
labels = {}
for c in cluster_profile.index:
    lt  = cluster_profile.loc[c, 'Avg_LeadTime']
    prf = cluster_profile.loc[c, 'Avg_Profit']
    if lt == cluster_profile['Avg_LeadTime'].max():
        labels[c] = 'Slow & Costly Routes'
    elif prf == cluster_profile['Avg_Profit'].max():
        labels[c] = 'High Value Routes'
    else:
        labels[c] = 'Standard Routes'

df['Cluster_Label'] = df['Cluster'].map(labels)

fig, ax = plt.subplots(figsize=(9, 5))
for c, label in labels.items():
    mask = df['Cluster'] == c
    ax.scatter(df.loc[mask,'Lead Time'], df.loc[mask,'Gross Profit'],
               alpha=0.3, s=20, label=label)
ax.set_xlabel('Lead Time (days)'); ax.set_ylabel('Gross Profit ($)')
ax.set_title('Route Clusters: Lead Time vs Profit', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('\nCluster Distribution by Factory:')
print(pd.crosstab(df['Factory'], df['Cluster_Label']))

---
## 🔁 Section 7: Scenario Simulation Engine

In [ ]:
# ── Simulate factory reassignment for any product ─────────
def simulate_reassignment(product_name, region='Atlantic', ship_mode='Standard Class',
                           units=3, country='United States'):
    """Predict lead time for a product across all factories."""
    current_factory = FACTORY_MAP.get(product_name)
    division        = df.loc[df['Product Name'] == product_name, 'Division'].iloc[0]
    results = []

    for factory in ALL_FACTORIES:
        row = {
            'Ship Mode_enc':     le_dict['Ship Mode'].transform([ship_mode])[0],
            'Region_enc':        le_dict['Region'].transform([region])[0],
            'Product Name_enc':  le_dict['Product Name'].transform([product_name])[0],
            'Factory_enc':       le_dict['Factory'].transform([factory])[0],
            'Division_enc':      le_dict['Division'].transform([division])[0],
            'Units':             units,
            'Country/Region_enc':le_dict['Country/Region'].transform([country])[0]
        }
        X_pred  = pd.DataFrame([row])[FEATURE_COLS]
        pred_lt = best_model.predict(X_pred)[0]

        results.append({
            'Factory':            factory,
            'Predicted Lead Time':round(pred_lt),
            'Is Current':         factory == current_factory
        })

    sim_df = pd.DataFrame(results).sort_values('Predicted Lead Time')
    current_lt = sim_df.loc[sim_df['Is Current'], 'Predicted Lead Time'].values[0]
    sim_df['Lead Time Reduction'] = (current_lt - sim_df['Predicted Lead Time'])
    sim_df['Status'] = sim_df['Factory'].apply(lambda x: '⭐ Current' if x == current_factory else '🔁 Alternative')
    return sim_df, current_factory

# ── Example simulation: Hair Toffee (worst bottleneck) ────
sim_df, curr = simulate_reassignment('Hair Toffee')
print(f"Simulation for: Hair Toffee | Current Factory: {curr}")
print(sim_df[['Status','Factory','Predicted Lead Time','Lead Time Reduction']].to_string(index=False))

In [ ]:
# ── Visualize scenario for Hair Toffee ────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e15759' if r['Is Current'] else '#4e79a7' for _, r in sim_df.iterrows()]

bars = ax.bar(sim_df['Factory'], sim_df['Predicted Lead Time'], color=colors)
red_patch  = mpatches.Patch(color='#e15759', label='Current Factory')
blue_patch = mpatches.Patch(color='#4e79a7', label='Alternative Factories')
ax.legend(handles=[red_patch, blue_patch])

for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')

ax.set_title('Scenario Simulation: Hair Toffee — Lead Time by Factory', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Lead Time (days)')
ax.set_xticklabels(sim_df['Factory'], rotation=20, ha='right')
plt.tight_layout()
plt.show()

# Simulate all products
print('\n📊 Simulation across ALL products (Atlantic, Standard Class):')
print(f'{"Product":<42} {"Current":>10} {"Best":>10} {"Best Factory":<20} {"Savings":>8}')
print('-' * 95)
for product in sorted(df['Product Name'].unique()):
    s_df, curr_fac = simulate_reassignment(product)
    curr_lt  = s_df.loc[s_df['Is Current'], 'Predicted Lead Time'].values[0]
    best_row = s_df.iloc[0]
    print(f'{product:<42} {curr_lt:>10} {best_row["Predicted Lead Time"]:>10} {best_row["Factory"]:<20} {best_row["Lead Time Reduction"]:>+8}')

---
## 🏆 Section 8: Optimization & Recommendations

In [ ]:
# ── Generate ranked recommendations ───────────────────────
recommendations = []

for product in df['Product Name'].unique():
    current_factory = FACTORY_MAP.get(product)
    division        = df.loc[df['Product Name'] == product, 'Division'].iloc[0]

    base_row = {
        'Ship Mode_enc':     le_dict['Ship Mode'].transform(['Standard Class'])[0],
        'Region_enc':        le_dict['Region'].transform(['Atlantic'])[0],
        'Product Name_enc':  le_dict['Product Name'].transform([product])[0],
        'Factory_enc':       le_dict['Factory'].transform([current_factory])[0],
        'Division_enc':      le_dict['Division'].transform([division])[0],
        'Units': 3,
        'Country/Region_enc':le_dict['Country/Region'].transform(['United States'])[0]
    }
    current_lt = best_model.predict(pd.DataFrame([base_row])[FEATURE_COLS])[0]

    best_lt = current_lt
    best_factory = current_factory
    for factory in ALL_FACTORIES:
        if factory == current_factory: continue
        row = dict(base_row)
        row['Factory_enc'] = le_dict['Factory'].transform([factory])[0]
        lt = best_model.predict(pd.DataFrame([row])[FEATURE_COLS])[0]
        if lt < best_lt:
            best_lt = lt; best_factory = factory

    avg_profit = df[df['Product Name'] == product]['Gross Profit'].mean()
    reduction  = current_lt - best_lt

    recommendations.append({
        'Product':             product,
        'Current Factory':     current_factory,
        'Recommended Factory': best_factory,
        'Lead Time Reduction': round(reduction),
        'Avg Profit/Order':    round(avg_profit, 2),
        'Priority Score':      round(reduction * 0.7 + avg_profit * 0.3, 1),
        'Action':              'Reassign' if best_factory != current_factory and reduction > 0 else 'Keep Current'
    })

rdf = pd.DataFrame(recommendations).sort_values('Priority Score', ascending=False).reset_index(drop=True)
rdf.index += 1
rdf.index.name = 'Rank'

print('🏆 TOP FACTORY REASSIGNMENT RECOMMENDATIONS')
print('=' * 90)
print(rdf[['Product','Current Factory','Recommended Factory',
           'Lead Time Reduction','Avg Profit/Order','Priority Score','Action']].to_string())

In [ ]:
# ── Recommendation visualizations ─────────────────────────
reassign_df = rdf[rdf['Action'] == 'Reassign'].head(8)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Lead time reduction
axes[0].barh(reassign_df['Product'], reassign_df['Lead Time Reduction'], color='#4e79a7')
axes[0].set_title('Lead Time Reduction Potential by Product', fontweight='bold')
axes[0].set_xlabel('Days Saved')
axes[0].tick_params(labelsize=8)
for i, v in enumerate(reassign_df['Lead Time Reduction']):
    axes[0].text(v+0.2, i, f'{v}d', va='center', fontsize=9)

# Priority score
axes[1].barh(reassign_df['Product'], reassign_df['Priority Score'], color='#f28e2b')
axes[1].set_title('Priority Score (Speed 70% + Profit 30%)', fontweight='bold')
axes[1].set_xlabel('Priority Score')
axes[1].tick_params(labelsize=8)
for i, v in enumerate(reassign_df['Priority Score']):
    axes[1].text(v+0.2, i, f'{v}', va='center', fontsize=9)

plt.suptitle('Factory Reassignment Recommendations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Lead Time vs Profit scatter (priority matrix)
combined = df.groupby('Product Name').agg(
    Avg_LT=('Lead Time','mean'), Avg_Profit=('Gross Profit','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(combined['Avg_LT'], combined['Avg_Profit'], s=120, c='#f28e2b', zorder=3, edgecolors='white', lw=0.5)
for _, row in combined.iterrows():
    ax.annotate(row['Product Name'], (row['Avg_LT'], row['Avg_Profit']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.axvline(combined['Avg_LT'].mean(), color='red', linestyle='--', alpha=0.6, label='Avg Lead Time')
ax.axhline(combined['Avg_Profit'].mean(), color='blue', linestyle='--', alpha=0.6, label='Avg Profit')
ax.text(combined['Avg_LT'].max()*0.97, combined['Avg_Profit'].max()*0.97,
        '🔴 HIGH PRIORITY\n(High LT + High Profit)', ha='right', fontsize=9,
        color='red', bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))
ax.set_xlabel('Avg Lead Time (days)', fontsize=11)
ax.set_ylabel('Avg Profit per Order ($)', fontsize=11)
ax.set_title('Lead Time vs Profit — Factory Reassignment Priority Matrix', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 📋 Section 9: KPI Summary & Final Insights

In [ ]:
# ── Final KPI Dashboard ───────────────────────────────────
reassign_count = len(rdf[rdf['Action'] == 'Reassign'])
max_saving     = rdf['Lead Time Reduction'].max()
top_product    = rdf.iloc[0]['Product']
top_recommend  = rdf.iloc[0]['Recommended Factory']

print('=' * 65)
print('  📊 FINAL KPI SUMMARY — Nassau Candy Distributor')
print('=' * 65)
print(f'  Total Orders Analyzed     : {len(df):,}')
print(f'  Total Revenue             : ${df["Sales"].sum():>12,.2f}')
print(f'  Total Gross Profit        : ${df["Gross Profit"].sum():>12,.2f}')
print(f'  Overall Profit Margin     : {df["Margin %"].mean():>11.1f}%')
print(f'  Avg Lead Time (Current)   : {df["Lead Time"].mean():>10.0f} days')
print(f'  Products to Reassign      : {reassign_count:>12}')
print(f'  Max Lead Time Saving      : {max_saving:>10} days')
print(f'  Top Priority Product      : {top_product}')
print(f'  Recommended Factory       : {top_recommend}')
print('=' * 65)

print('''
🔍 KEY INSIGHTS:

1. BOTTLENECK ROUTES:
   → Hair Toffee (The Other Factory) has the highest avg lead time
   → Everlasting Gobstopper & SweeTARTS are the next critical bottlenecks

2. SHIPPING MODE INEFFICIENCY:
   → Standard Class is FASTER than First Class — a counterintuitive finding
   → Nassau should audit premium shipping lanes for routing inefficiencies

3. FACTORY OVERLOAD:
   → Lot's O' Nuts handles 56% of all orders (5,692 orders)
   → The Other Factory handles only 100 orders but has the BEST lead time (1,280 days)
   → Opportunity: redistribute volume to The Other Factory

4. HIGH VALUE + HIGH DELAY PRODUCTS (Top Reassignment Candidates):
   → Lickable Wallpaper: $41.81 avg profit but 1,339 day lead time
   → Everlasting Gobstopper: $34.67 avg profit but 1,395 day lead time
   → These generate the most revenue impact if lead times are reduced

5. REGIONAL PERFORMANCE:
   → Gulf region has the best delivery performance (1,311 days avg)
   → All regions are relatively close — systemic improvement possible

💡 RECOMMENDATIONS:
   1. Reassign Hair Toffee to a factory with lower avg lead time
   2. Investigate and fix First Class shipping lanes
   3. Shift some Lot's O' Nuts volume to The Other Factory
   4. Prioritize lead time reduction for high-profit products
   5. Focus optimization efforts on Gulf region best practices
''')

In [ ]:
# ── Final visual summary ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Nassau Candy Distributor — Executive Summary Dashboard',
             fontsize=14, fontweight='bold')

# 1. Factory lead time comparison
fac_lt = df.groupby('Factory')['Lead Time'].mean().sort_values()
colors_lt = ['#59a14f' if v == fac_lt.min() else '#e15759' if v == fac_lt.max() else '#4e79a7' for v in fac_lt]
axes[0,0].barh(fac_lt.index, fac_lt.values, color=colors_lt)
axes[0,0].set_title('Avg Lead Time by Factory\n(🟢 Best | 🔴 Worst)', fontweight='bold')
axes[0,0].set_xlabel('Days')
for i, v in enumerate(fac_lt.values):
    axes[0,0].text(v+1, i, f'{v:.0f}d', va='center', fontsize=9)

# 2. Profit contribution
fac_profit = df.groupby('Factory')['Gross Profit'].sum().sort_values(ascending=False)
axes[0,1].pie(fac_profit.values, labels=fac_profit.index, autopct='%1.1f%%',
              colors=['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f'])
axes[0,1].set_title('Profit Share by Factory', fontweight='bold')

# 3. Top recommendations
top_recs = rdf[rdf['Action']=='Reassign'].head(6)
axes[1,0].barh(top_recs['Product'], top_recs['Lead Time Reduction'], color='#f28e2b')
axes[1,0].set_title('Top 6: Lead Time Reduction Potential', fontweight='bold')
axes[1,0].set_xlabel('Days Saved')
axes[1,0].tick_params(labelsize=8)

# 4. Profitability by division
div_profit = df.groupby('Division')['Gross Profit'].sum()
axes[1,1].pie(div_profit.values, labels=div_profit.index, autopct='%1.1f%%',
              colors=['#4e79a7','#f28e2b','#59a14f'])
axes[1,1].set_title('Profit Share by Division', fontweight='bold')

plt.tight_layout()
plt.savefig('nassau_executive_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Executive Summary chart saved as nassau_executive_summary.png')

---

## ✅ Conclusion

This project elevated Nassau Candy Distributor from **descriptive analytics to intelligent decision-making** by:

- **Predicting** shipping lead times using Machine Learning (Linear Regression, Random Forest, Gradient Boosting)
- **Clustering** routes to identify slow and congested shipping lanes
- **Simulating** factory reassignment scenarios to quantify operational impact
- **Recommending** optimal factory-product configurations ranked by lead time reduction and profit impact

The system provides **actionable factory reallocation recommendations** that can improve shipping efficiency without sacrificing profitability.

---
*Project by: [Your Name] | Internship Project | Data Analytics — Supply Chain Domain*